# FLUT: current collected data

Start here for the September 12 rebuild. Inspect source coverage, recent FanDuel observations, and New York weekly activity. Amounts are USD.

This is a fresh, unreviewed capture of currently available official history. The original approved studies remain separate. Retrieval today does not establish historical public availability. No forecast or valuation is changed here.


In [ ]:
database_file = "data/staging/refresh_20260912T201854Z/gaming_current.sqlite"
ny_weeks = 8
report_age_warning_days = 90  # review threshold, not a publication deadline


In [ ]:
from pathlib import Path
import hashlib
import json
import sqlite3
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if (ROOT / "gaming" / "src").is_dir():
    ROOT = ROOT / "gaming"
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
DB = ROOT / database_file
RUN = DB.parent
manifest = json.loads((RUN / "run_manifest.json").read_text())
validation = json.loads((RUN / "validation.json").read_text())
if not manifest.get("finished_at"):
    raise RuntimeError("Collection is still running; inspect its log first.")
before = hashlib.sha256(DB.read_bytes()).hexdigest()
if before != validation["database_sha256"]:
    raise RuntimeError("Database differs from the validated capture; review the new run before use.")
with sqlite3.connect(DB.resolve().as_uri() + "?mode=ro", uri=True) as connection:
    rows = pd.read_sql_query("SELECT * FROM gaming_results", connection)
    coverage = pd.read_sql_query("SELECT * FROM source_coverage", connection)
summary = pd.read_csv(RUN / "collection_summary.csv")
pd.set_option("display.max_colwidth", None)
print("Database:", DB)
print("Capture finished:", manifest["finished_at"])
print("Rows:", len(rows), "| Source files verified:", validation["verified_source_files"])
print("Research status: unreviewed current capture; no approved forecast mapping.")


## What is available, and what failed?

Dates are reporting periods. A completed collector can still have partial coverage. Missing periods and unsupported products do not mean zero activity. Source definitions are in `config/state_metric_notes.csv`; see notebook 90 to explore a particular metric.

Observation age is measured at this capture date, not from an inferred publication date. The configurable 90-day flag identifies older series even when collection itself succeeded.

A selected recent refresh does not recheck all history. Other series keep their prior coverage and are labelled not_selected_this_run.


In [ ]:
counts = rows.groupby(["state_code", "vertical"], as_index=False).agg(
    observations=("operator", "size"), first_period=("period_start", "min"),
    last_period=("period_end", "max"), source_versions=("source_sha256", "nunique"))
capture_date = pd.Timestamp(manifest["finished_at"]).tz_convert("UTC").tz_localize(None).normalize()
counts["period_age_days"] = (capture_date - pd.to_datetime(counts.last_period)).dt.days
status = coverage[["state_code", "vertical", "status", "reason"]].rename(columns={"status": "retained_coverage", "reason": "retained_reason"})
status = status.merge(summary[["state_code", "vertical", "run_status", "coverage_status", "coverage_reason"]], on=["state_code", "vertical"], how="outer")
status["run_status"] = status.run_status.fillna("not_selected_this_run")
status["coverage_status"] = status.coverage_status.fillna(status.retained_coverage)
status["coverage_reason"] = status.coverage_reason.fillna(status.retained_reason)
status = status.drop(columns=["retained_coverage", "retained_reason"])
status = status.merge(counts, on=["state_code", "vertical"], how="outer").sort_values(["state_code", "vertical"])
display(status)
print("Sources requiring attention (coverage exceptions or older observations):")
display(status[~status.run_status.isin(["completed", "not_selected_this_run"]) | ~status.coverage_status.isin(["ok", "recent_only"])
               | status.period_age_days.gt(report_age_warning_days)])


## Latest observations explicitly labelled FanDuel

Each state/product retains its own reporting period and measure. This name filter does not resolve unbranded licensees or imply complete national FanDuel coverage. Gross revenue, adjusted revenue, and taxable revenue remain separate. Some legacy generic source labels describe only one reported measure; use the named numeric column and verify the report.


In [ ]:
fanduel = rows[rows.operator.astype(str).str.contains("fanduel", case=False, regex=False)].copy()
latest = fanduel.groupby(["state_code", "vertical", "channel", "frequency"])["period_end"].transform("max")
latest_fanduel = fanduel[fanduel.period_end.eq(latest)].sort_values(["state_code", "vertical", "operator"])
display(latest_fanduel[["state_code", "vertical", "channel", "frequency", "operator", "period_end",
                       "handle", "gross_revenue", "adjusted_revenue", "taxable_revenue",
                       "reported_revenue_name", "source_file", "source_url"]])


## New York: recent complete weeks

New York reports cash-basis sportsbook GGR. Weekly numbers remain weekly. Handle share is descriptive and cannot identify customer migration, promotion costs, or company net revenue. Seasonal sports schedules make simple week-to-week growth difficult to interpret.


In [ ]:
ny = rows[rows.state_code.eq("NY") & rows.frequency.eq("weekly")].copy()
fd = ny[ny.operator.eq("FanDuel") & ny.row_type.eq("operator")]
market = ny[ny.row_type.eq("official_statewide_total")]
if fd.empty or market.empty:
    print("New York operator/statewide evidence is unavailable in this capture.")
else:
    keys = ["period_start", "period_end"]
    joined = fd.merge(market, on=keys, suffixes=("_fd", "_market"), validate="one_to_one").sort_values("period_end").tail(ny_weeks)
    operators = ny[ny.row_type.eq("operator")].groupby(keys).agg(
        operator_handle=("handle", lambda s: s.sum(min_count=len(s))),
        operator_ggr=("gross_revenue", lambda s: s.sum(min_count=len(s)))).reset_index()
    joined = joined.merge(operators, on=keys, validate="one_to_one")
    reconciled = ((joined.handle_market - joined.operator_handle).abs().le(0.01)
                  & (joined.gross_revenue_market - joined.operator_ggr).abs().le(0.01))
    if not reconciled.all() or joined.empty:
        raise ValueError("Recent NY operator rows do not reconcile to the printed statewide totals.")
    if not (joined.handle_market.gt(0) & joined.handle_fd.ge(0) & joined.handle_fd.le(joined.handle_market)).all():
        raise ValueError("NY handle share requires valid nonnegative operator/market handles.")
    recent = pd.DataFrame({"week_ended": joined.period_end, "FanDuel_handle": joined.handle_fd,
        "statewide_handle": joined.handle_market, "FanDuel_handle_share_pct": 100 * joined.handle_fd / joined.handle_market,
        "FanDuel_GGR": joined.gross_revenue_fd, "statewide_GGR": joined.gross_revenue_market})
    display(recent)
    print("Printed statewide totals reconcile within one cent for every displayed week.")
    display(joined[["period_end", "source_url_fd", "source_url_market"]])


In [ ]:
assert hashlib.sha256(DB.read_bytes()).hexdigest() == before
print("Read-only review complete. Database bytes are unchanged.")
